In [1]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

In [2]:

app = dash.Dash('MLFinance')

app.layout = html.Div([
    html.H1("Анализ котировок", style={'textAlign': 'center'}),
    dcc.Graph(id='combined-chart'),
    dcc.Interval(
        id='interval-component',
        interval=60*1000,  # Обновление раз в минуту
        n_intervals=0
    )
])

In [3]:


@app.callback(
    Output('combined-chart', 'figure'),
    [Input('interval-component', 'n_intervals')]
)
def update_graph(n):
    global streaming_data
    df=streaming_data.tail(100)

    fig = make_subplots(
        rows=2, 
        cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.05,
        row_heights=[0.7, 0.3]
    )
    
    # Добавление свечного графика
    fig.add_trace(
        go.Candlestick(
            x=df['time'],
            open=df['open'],
            high=df['high'],
            low=df['low'],
            close=df['close'],
            name='Свечи',
            increasing_line_color='#2ecc71', 
            decreasing_line_color='#e74c3c',
            line=dict(width=2),
            xperiod=15,
            xperiodalignment="middle"
        ), 
        row=1, 
        col=1
    )
    
    # Маркеры аномалий цены
    price_anomalies = df[df['anomalies_price'] == 1]
    fig.add_trace(
        go.Scatter(
            x=price_anomalies['time'],
            y=price_anomalies['high'] * 1.005,
            mode='markers',
            marker=dict(
                color='#f1c40f',
                size=10,
                symbol='diamond',
                line=dict(width=1, color='black')
            ),
            name='Аномалии цены'
        ), 
        row=1, 
        col=1
    )
    

    volume_anomalies = df[df['anomalies_volume'] == 1]
    fig.add_trace(
        go.Scatter(
            x=volume_anomalies['time'],
            y=volume_anomalies['volume'] * 1.2,
            mode='markers',
            marker=dict(
                color='#3498db',
                size=12,
                symbol='diamond',
                line=dict(width=2)
            ),
            name='Аномалии объема'
        ), 
        row=2, 
        col=1
    )
    
    # Добавление гистограммы объема
    fig.add_trace(
        go.Bar(
            x=df['time'],
            y=df['volume'],
            name='Объем',
            marker_color='white',
            opacity=0.9,
            marker_line=dict(
            color='rgba(0,0,0,0.6)',  
            width=4
            ),
            width=10
        ),
        row=2,
        col=1
    )

    #last_time = df['time'].max()

    # Настройка внешнего вида
    fig.update_layout(
      #  title=f'Последнее обновление: {last_time.strftime("%Y-%m-%d %H:%M:%S")}',
        template='seaborn',
        height=800,
        hovermode='x unified',
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    # Настройка осей
    fig.update_xaxes(title_text='Время', row=2, col=1)
    fig.update_yaxes(title_text='Цена', row=1, col=1)
    fig.update_yaxes(title_text='Объем', row=2, col=1)
    
    # Отключение ползунка диапазона
    fig.update(layout_xaxis_rangeslider_visible=False)
    
    return fig

